In [ ]:
# CNN

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

# device config
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = torchvision.datasets.MNIST(root='./data',train=True, transform=transform,download=True)
test_dataset = torchvision.datasets.MNIST(root='./data',train=False,transform=transform,download=True)


train_loader = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=64, shuffle=False)

# define cnn model
class CNN(nn.Module):
  def __init__(self):
    super(CNN, self).__init__()
    self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1)
    self.conv2 = nn.Conv2d(in_channels=32,out_channels=64, kernel_size=3,padding=1)
    self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
    self.fc1 = nn.Linear(64 * 7 * 7 , 128)
    self.fc2 = nn.Linear(128 , 10)
    self.relu = nn.ReLU()


  def forward(self, x):
    x = self.pool(self.relu(self.conv1(x)))
    x = self.pool(self.relu(self.conv2(x)))
    x = x.view(x.size(0) , -1)
    x = self.relu(self.fc1(x))
    x = self.fc2(x)
    return x


# isntatiate model
model = CNN().to(device)


# define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(),lr=0.001)

epochs = 5
for epoch in range(epochs):
  model.train()
  for images, labels in train_loader:
    images, labels = images.to(device), labels.to(device)

    optimizer.zero_grad()
    outputs = model(images)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()

  print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

print("Training complete!")





Epoch [1/5], Loss: 0.0370
Epoch [2/5], Loss: 0.0740
Epoch [3/5], Loss: 0.0028
Epoch [4/5], Loss: 0.0055
Epoch [5/5], Loss: 0.0058
Training complete!


# Faster R-CNN adapted for MNIST !

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.models.detection import fasterrcnn_resnet50_fpn

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load and preprocess MNIST dataset
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = torchvision.datasets.MNIST(root='./data', train=True, transform=transform, download=True)
test_dataset = torchvision.datasets.MNIST(root='./data', train=False, transform=transform, download=True)

train_loader = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=64, shuffle=False)

# Define CNN Model
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# Instantiate model
model = CNN().to(device)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
epochs = 5
for epoch in range(epochs):
    model.train()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

    print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

print("Training complete!")

# Evaluate CNN model
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f'Test Accuracy of CNN Model: {accuracy:.2f}%')

# Adapt MNIST for Faster R-CNN (Bounding Boxes)
def mnist_to_object_detection(dataset):
    object_detection_data = []
    for img, label in dataset:
        boxes = torch.tensor([[5, 5, 23, 23]], dtype=torch.float32)  # Example fixed bounding box
        labels = torch.tensor([label], dtype=torch.int64)
        object_detection_data.append((img, {"boxes": boxes, "labels": labels}))
    return object_detection_data

train_dataset_od = mnist_to_object_detection(train_dataset)
test_dataset_od = mnist_to_object_detection(test_dataset)

# Load Faster R-CNN Model
faster_rcnn = fasterrcnn_resnet50_fpn(pretrained=True)
num_classes = 11  # 10 digits + background class
in_features = faster_rcnn.roi_heads.box_predictor.cls_score.in_features
faster_rcnn.roi_heads.box_predictor = nn.Linear(in_features, num_classes)
faster_rcnn.to(device)

print("Faster R-CNN model adapted for MNIST classification.")


Epoch [1/5], Loss: 0.0703
Epoch [2/5], Loss: 0.0097
Epoch [3/5], Loss: 0.0011
Epoch [4/5], Loss: 0.0056
Epoch [5/5], Loss: 0.0606
Training complete!
Test Accuracy of CNN Model: 99.16%


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=FasterRCNN_ResNet50_FPN_Weights.COCO_V1`. You can also use `weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth
100%|██████████| 160M/160M [00:02<00:00, 78.6MB/s]


Faster R-CNN model adapted for MNIST classification.


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.models.detection import fasterrcnn_resnet50_fpn

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load and preprocess MNIST dataset
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = torchvision.datasets.MNIST(root='./data', train=True, transform=transform, download=True)
test_dataset = torchvision.datasets.MNIST(root='./data', train=False, transform=transform, download=True)

train_loader = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=64, shuffle=False)

# Define CNN Model
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# Instantiate model
model = CNN().to(device)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
epochs = 5
for epoch in range(epochs):
    model.train()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
    print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

print("CNN Training complete!")

# Evaluate CNN model
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f'Test Accuracy of CNN Model: {accuracy:.2f}%')

# Adapt MNIST for Faster R-CNN (Bounding Boxes)
def mnist_to_object_detection(dataset):
    object_detection_data = []
    for img, label in dataset:
        boxes = torch.tensor([[5, 5, 23, 23]], dtype=torch.float32)  # Example fixed bounding box
        labels = torch.tensor([label], dtype=torch.int64)
        object_detection_data.append((img, {"boxes": boxes, "labels": labels}))
    return object_detection_data

train_dataset_od = mnist_to_object_detection(train_dataset)
test_dataset_od = mnist_to_object_detection(test_dataset)

# Load Faster R-CNN Model
faster_rcnn = fasterrcnn_resnet50_fpn(pretrained=True)
num_classes = 11  # 10 digits + background class
in_features = faster_rcnn.roi_heads.box_predictor.cls_score.in_features
faster_rcnn.roi_heads.box_predictor = nn.Linear(in_features, num_classes)
faster_rcnn.to(device)

# Define Faster R-CNN optimizer
faster_rcnn_optimizer = optim.Adam(faster_rcnn.parameters(), lr=0.001)

# Training loop for Faster R-CNN
epochs = 5
for epoch in range(epochs):
    faster_rcnn.train()
    for images, targets in train_dataset_od:
        images = images.unsqueeze(0).to(device)
        targets = [{key: val.to(device) for key, val in targets.items()}]
        faster_rcnn_optimizer.zero_grad()
        loss_dict = faster_rcnn(images, targets)
        loss = sum(loss for loss in loss_dict.values())
        loss.backward()
        faster_rcnn_optimizer.step()
    print(f'Faster R-CNN Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

print("Faster R-CNN training complete!")


100%|██████████| 9.91M/9.91M [00:00<00:00, 36.9MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.17MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 10.0MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 5.66MB/s]


Epoch [1/5], Loss: 0.0173
Epoch [2/5], Loss: 0.0913
Epoch [3/5], Loss: 0.0950
Epoch [4/5], Loss: 0.0116
Epoch [5/5], Loss: 0.0037
CNN Training complete!
Test Accuracy of CNN Model: 99.12%


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=FasterRCNN_ResNet50_FPN_Weights.COCO_V1`. You can also use `weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth
100%|██████████| 160M/160M [00:02<00:00, 82.4MB/s]


ValueError: too many values to unpack (expected 2)

In [7]:
# Evaluate Faster R-CNN
faster_rcnn.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, targets in test_dataset_od:
        images = images.unsqueeze(0).to(device)  # Add batch dimension

        # Get model predictions (DO NOT pass targets during evaluation)
        outputs = faster_rcnn(images)

        if outputs and 'labels' in outputs[0]:  # Ensure valid output
            predicted_labels = outputs[0]['labels'].cpu().numpy()  # Extract predicted labels
            true_label = targets["labels"].item()  # Extract true label

            if true_label in predicted_labels:
                correct += 1
        total += 1

accuracy_frcnn = 100 * correct / total if total > 0 else 0
print(f'Test Accuracy of Faster R-CNN Model: {accuracy_frcnn:.2f}%')


ValueError: too many values to unpack (expected 2)

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.models.detection import fasterrcnn_resnet50_fpn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = torchvision.datasets.MNIST(root='./data', train=True, transform=transform, download=True)
test_dataset = torchvision.datasets.MNIST(root='./data', train=False, transform=transform, download=True)

train_loader = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=64, shuffle=False)

# Define CNN Model
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# Instantiate model
model = CNN().to(device)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Evaluate Models After Training
print("\n=== Model Comparison ===")
print(f'CNN Accuracy: {accuracy:.2f}%')
print(f'Faster R-CNN Accuracy: {accuracy_frcnn:.2f}%')
if accuracy > accuracy_frcnn:
    print("CNN performed better in classification.")
elif accuracy < accuracy_frcnn:
    print("Faster R-CNN performed better in classification.")
else:
    print("Both models have the same classification accuracy.")





=== Model Comparison ===
CNN Accuracy: 99.12%


NameError: name 'accuracy_frcnn' is not defined

In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.models import vgg16, alexnet
from torchvision.models.detection import fasterrcnn_resnet50_fpn

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load and preprocess MNIST dataset
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),  # Convert MNIST to 3 channels for VGG16 & AlexNet
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = torchvision.datasets.MNIST(root='./data', train=True, transform=transform, download=True)
test_dataset = torchvision.datasets.MNIST(root='./data', train=False, transform=transform, download=True)

train_loader = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=64, shuffle=False)

# Fine-tune VGG16
vgg_model = vgg16(pretrained=True)
vgg_model.classifier[6] = nn.Linear(4096, 10)  # Modify final layer for 10 classes
vgg_model = vgg_model.to(device)
vgg_optimizer = optim.Adam(vgg_model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

# Train VGG16
epochs = 5
for epoch in range(epochs):
    vgg_model.train()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        vgg_optimizer.zero_grad()
        outputs = vgg_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        vgg_optimizer.step()
    print(f'VGG16 Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

# Evaluate VGG16
vgg_model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = vgg_model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy_vgg = 100 * correct / total
print(f'Test Accuracy of VGG16 Model: {accuracy_vgg:.2f}%')

# Fine-tune AlexNet
alex_model = alexnet(pretrained=True)
alex_model.classifier[6] = nn.Linear(4096, 10)  # Modify final layer for 10 classes
alex_model = alex_model.to(device)
alex_optimizer = optim.Adam(alex_model.parameters(), lr=0.001)

# Train AlexNet
epochs = 5
for epoch in range(epochs):
    alex_model.train()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        alex_optimizer.zero_grad()
        outputs = alex_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        alex_optimizer.step()
    print(f'AlexNet Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

# Evaluate AlexNet
alex_model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = alex_model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy_alex = 100 * correct / total
print(f'Test Accuracy of AlexNet Model: {accuracy_alex:.2f}%')

# Compare All Models
print("\n=== Model Comparison ===")
print(f'CNN Accuracy: {accuracy:.2f}%')
print(f'Faster R-CNN Accuracy: {accuracy_frcnn:.2f}%')
print(f'VGG16 Accuracy: {accuracy_vgg:.2f}%')
print(f'AlexNet Accuracy: {accuracy_alex:.2f}%')


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth
100%|██████████| 528M/528M [00:07<00:00, 77.2MB/s]


RuntimeError: Given input size: (512x1x1). Calculated output size: (512x0x0). Output size is too small

In [11]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.models import vgg16, alexnet
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from transformers import ViTForImageClassification, ViTFeatureExtractor

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load and preprocess MNIST dataset
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),  # Convert MNIST to 3 channels for ViT
    transforms.Resize((224, 224)),  # Resize for ViT input
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = torchvision.datasets.MNIST(root='./data', train=True, transform=transform, download=True)
test_dataset = torchvision.datasets.MNIST(root='./data', train=False, transform=transform, download=True)

train_loader = DataLoader(dataset=train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=32, shuffle=False)

# Load Vision Transformer (ViT) model
# The pretrained model has a classifier head for 1000 classes.
# We replace it with a new classifier head for 10 classes (MNIST).
vit_model = ViTForImageClassification.from_pretrained("google/vit-base-patch16-224", num_labels=10)
vit_model.classifier = nn.Linear(vit_model.config.hidden_size, 10) # Replace the classifier head
vit_model = vit_model.to(device)

vit_optimizer = optim.Adam(vit_model.parameters(), lr=0.0001)
criterion = nn.CrossEntropyLoss()

# Train ViT model
epochs = 5
for epoch in range(epochs):
    vit_model.train()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        vit_optimizer.zero_grad()
        outputs = vit_model(images).logits  # Extract logits
        loss = criterion(outputs, labels)
        loss.backward()
        vit_optimizer.step()
    print(f'ViT Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

# Evaluate ViT model
vit_model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = vit_model(images).logits
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy_vit = 100 * correct / total
print(f'Test Accuracy of Vision Transformer (ViT) Model: {accuracy_vit:.2f}%')

# Compare All Models
print("\n=== Model Comparison ===")
print(f'CNN Accuracy: {accuracy:.2f}%')
print(f'Faster R-CNN Accuracy: {accuracy_frcnn:.2f}%')
print(f'VGG16 Accuracy: {accuracy_vgg:.2f}%')
print(f'AlexNet Accuracy: {accuracy_alex:.2f}%')
print(f'ViT Accuracy: {accuracy_vit:.2f}%')

RuntimeError: Error(s) in loading state_dict for ViTForImageClassification:
	size mismatch for classifier.weight: copying a param with shape torch.Size([1000, 768]) from checkpoint, the shape in current model is torch.Size([10, 768]).
	size mismatch for classifier.bias: copying a param with shape torch.Size([1000]) from checkpoint, the shape in current model is torch.Size([10]).

SyntaxError: invalid decimal literal (<ipython-input-12-93b2f253adee>, line 15)